In [1]:
import sys

import torch

import pandas as pd

from config.feature_config import FeatureConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from dice4el.scenario.scenario_handler import ScenarioHandler

from dice4el.scenario.scenario_model import ScenarioLSTM
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper

from dice4el.dice4el_config import EventLogDiCEConfig
from dice4el.eventlog_dice import EventLogDiCE
from dice4el.eventlog_dice_optimized import EventLogDiCEOptimized

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=42)

In [3]:
df = pd.read_excel(
    "../../../data/sepsis.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:group": "string",
        "case:age": "float32",
        "Leucocytes": "float32",
        "CRP": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,CRP,Leucocytes,case:age,concept:name,lifecycle:transition,org:group,time_delta
0,A,2014-10-22 11:15:41,0.0,0.0,85.0,ER Registration,complete,A,0
1,A,2014-10-22 11:27:00,0.0,9.6,85.0,Leucocytes,complete,B,679
2,A,2014-10-22 11:27:00,21.0,0.0,85.0,CRP,complete,B,0
3,A,2014-10-22 11:27:00,0.0,0.0,85.0,LacticAcid,complete,B,0
4,A,2014-10-22 11:33:37,0.0,0.0,85.0,ER Triage,complete,C,397
5,A,2014-10-22 11:34:00,0.0,0.0,85.0,ER Sepsis Triage,complete,A,23
6,A,2014-10-22 14:03:47,0.0,0.0,85.0,IV Liquid,complete,A,8987
7,A,2014-10-22 14:03:47,0.0,0.0,85.0,IV Antibiotics,complete,A,0
8,A,2014-10-22 14:13:19,0.0,0.0,85.0,Admission NC,complete,D,572
9,A,2014-10-24 09:00:00,109.0,0.0,85.0,CRP,complete,B,154001


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['CRP', 'Leucocytes', 'case:age', 'concept:name', 'lifecycle:transition', 'org:group', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
org:group                      categorical    event    yes    ['A', 'B', 'C', ...]                     N/A        data_derived        
case:age                       continuous     case     yes    [40.00, 90.00]                           10.0000    quantile_derived    
time_delta                     continuous     event    yes    [0.00, 38748.80]                         139.0000   quantile_derived    
Leucocytes                    

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

In [11]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [12]:
scenario_model = ScenarioLSTM.load()

In [13]:
scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

### --- Process Constraints ---

In [14]:
engine = ProcessModelConstraintEngine.load(
     path = "../pretrained_models/"
)

In [15]:
engine.parallel_sets

[{'Admission NC',
  'CRP',
  'ER Registration',
  'ER Sepsis Triage',
  'ER Triage',
  'IV Antibiotics',
  'IV Liquid',
  'LacticAcid',
  'Leucocytes'},
 {'ER Registration', 'ER Sepsis Triage', 'ER Triage', 'IV Antibiotics'}]

In [16]:
engine.branching_sets

[{'Admission NC',
  'CRP',
  'ER Registration',
  'ER Sepsis Triage',
  'ER Triage',
  'IV Antibiotics',
  'IV Liquid',
  'LacticAcid',
  'Leucocytes'},
 {'ER Registration', 'ER Sepsis Triage', 'ER Triage', 'IV Antibiotics'},
 {'Release C', 'Release D', 'Release E'}]

### --- Load Experiments ---

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/sepsis-cf_generated_experiments_dice4el_output.txt", console=False)

In [18]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [19]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [20]:
dice4el_config = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
    w_margin_loss=1.0,
    w_scenario_loss=1.0,
    w_distance_loss=1.0,
    w_cat_loss=1.0,
)
dice4el_config.validate()

In [21]:
cf_DiCE4EL = EventLogDiCE(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results = generator.run_experiment_df(
    cf_method=cf_DiCE4EL,
    technique="DiCE4EL",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/190 [00:00<?, ?case/s]

In [22]:
results

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,FFA,3,1,3,0.208907,0.417813,0.0,0.777778,0.333333,...,2.156051,0.333333,0.208907,0.0,0.417813,0.777778,0.836033,0.836033,1.000000,1.000000
1,0,CL,4,1,4,0.163231,0.326462,0.0,0.777778,0.272727,...,2.048884,0.272727,0.163231,0.0,0.326462,0.777778,0.835148,0.835148,1.000000,1.000000
2,0,KY,5,1,5,0.169708,0.339415,0.0,0.666667,0.230769,...,1.903279,0.230769,0.169708,0.0,0.339415,0.666667,0.836136,0.836136,1.000000,1.000000
3,0,FP,6,1,6,0.127801,0.255601,0.0,0.777778,0.333333,...,2.079761,0.333333,0.127801,0.0,0.255601,0.777778,0.840849,0.840849,1.000000,1.000000
4,0,GIA,7,1,7,0.185711,0.371422,0.0,0.777778,0.294118,...,2.093697,0.294118,0.185711,0.0,0.371422,0.777778,0.836090,0.836090,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,18,MFA,12,1,11,0.119334,0.238668,0.0,0.733333,0.555556,...,1.408223,0.555556,0.119334,0.0,0.238668,0.733333,0.000000,0.000000,0.000000,0.000000
163,18,OKA,12,1,11,0.115845,0.231690,0.0,0.711111,0.259259,...,1.086216,0.259259,0.115845,0.0,0.231690,0.711111,0.000000,0.350262,0.000000,0.000000
164,18,SS,12,1,11,0.122496,0.244991,0.0,0.733333,0.185185,...,1.041014,0.185185,0.122496,0.0,0.244991,0.733333,0.000000,0.747166,0.000000,0.999999
165,18,ZFA,12,1,11,0.111194,0.222389,0.0,0.733333,0.185185,...,1.669781,0.185185,0.111194,0.0,0.222389,0.733333,0.640068,0.640068,0.999998,0.999998


In [23]:
cf_DiCE4EL_optim = EventLogDiCEOptimized(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results_optim = generator.run_experiment_df(
    cf_method=cf_DiCE4EL_optim,
    technique="DiCE4EL-Optimized",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/190 [00:00<?, ?case/s]

In [24]:
results_optim

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,FFA,3,1,3,0.404422,0.308844,0.500000,0.777778,0.555556,...,2.303911,0.555556,0.404422,0.500000,0.308844,0.777778,0.566155,0.566155,0.999995,0.999995
1,0,CL,4,1,4,0.407188,0.314376,0.500000,0.777778,0.181818,...,1.366784,0.181818,0.407188,0.500000,0.314376,0.777778,0.000000,0.000000,0.000000,0.000000
2,0,KY,5,1,5,0.364690,0.229380,0.500000,0.777778,0.692308,...,2.409183,0.692308,0.364690,0.500000,0.229380,0.777778,0.574407,0.574407,0.999996,0.999996
3,0,FP,6,1,6,0.382564,0.265129,0.500000,0.777778,0.400000,...,1.560342,0.400000,0.382564,0.500000,0.265129,0.777778,0.000000,0.000000,0.000000,0.000000
4,0,GIA,7,1,7,0.368443,0.236886,0.500000,0.777778,0.764706,...,2.440579,0.764706,0.368443,0.500000,0.236886,0.777778,0.529652,0.529652,0.999988,0.999988
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,18,MFA,12,1,11,0.123496,0.246991,0.000000,0.733333,0.555556,...,1.412385,0.555556,0.123496,0.000000,0.246991,0.733333,0.000000,0.000000,0.000000,0.000000
163,18,OKA,12,1,11,0.129476,0.258953,0.000000,0.733333,0.259259,...,1.122069,0.259259,0.129476,0.000000,0.258953,0.733333,0.000000,0.000000,0.000000,0.000000
164,18,SS,12,1,11,0.399805,0.345066,0.454545,0.822222,0.777778,...,1.999805,0.777778,0.399805,0.454545,0.345066,0.822222,0.000000,0.000000,0.000000,0.000000
165,18,ZFA,12,1,11,0.389813,0.325081,0.454545,0.844444,0.407407,...,1.641665,0.407407,0.389813,0.454545,0.325081,0.844444,0.000000,0.000000,0.000000,0.000000


### --- Cleanup ---

In [25]:
sys.stdout = original_stdout
log_file.close()